# 1.3 — División en splits train / val / test

Divide el dataset escalado en tres conjuntos estratificados:
70 % train, 15 % val, 15 % test (ajustado desde config).
Genera `data/processed/train.csv`, `val.csv` y `test.csv`.
Verifica que no hay fuga de datos entre splits y muestra la distribución por clase.

**Autor:** Jorge Ceferino Valdez — Maestría en Informática y Sistemas (UNPA)

In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

In [ ]:
project_root = Path(os.path.abspath('')).parent
if project_root not in sys.path:
    sys.path.insert(0, str(project_root))
print(f'project_root: {project_root}')

In [ ]:
from src.config import config, external_datasets, project_dir, interim_data_dir, processed_data_dir

# Semilla desde config
seed = config['data']['random_seed']
val_size  = config['data']['val_size']   # 0.15
test_size = config['data']['test_size']  # 0.10 (sobre total; ajustado abajo)

random.seed(seed)
np.random.seed(seed)
print(f'Semilla: {seed} | val_size: {val_size} | test_size: {test_size}')

In [ ]:
# Directorio de entrada: imágenes escaladas
base_dir = interim_data_dir() / 'dataset_scaled'
print(f'Cargando imágenes desde: {base_dir}')
assert base_dir.exists(), f'Ejecutar notebook 1.2 primero: {base_dir}'

## 1. Recolección de rutas e etiquetas

In [ ]:
image_paths = []
labels = []

for class_dir in sorted(base_dir.iterdir()):
    if not class_dir.is_dir():
        continue
    for img_file in sorted(class_dir.iterdir()):
        if img_file.suffix.lower() in ('.png', '.jpg', '.jpeg'):
            image_paths.append(str(img_file))
            labels.append(class_dir.name)

print(f'Total de imágenes: {len(image_paths)}')
print(f'Clases encontradas: {sorted(set(labels))}')
print(f'Distribución global: {dict(sorted(Counter(labels).items()))}')

## 2. División estratificada

In [ ]:
# Paso 1: separar test del resto
# test_size sobre el total; val_size sobre el resto
temp_size = val_size + test_size  # fracción combinada val+test

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths, labels,
    test_size=temp_size,
    stratify=labels,
    random_state=seed
)

# Paso 2: dividir temp en val y test en partes iguales
val_fraction = val_size / temp_size  # fracción de val dentro del bloque temp

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    test_size=0.5,  # val_size == test_size → 50/50 del bloque temp
    stratify=temp_labels,
    random_state=seed
)

print(f'Train: {len(train_paths)} ({len(train_paths)/len(image_paths)*100:.1f}%)')
print(f'Val:   {len(val_paths)}   ({len(val_paths)/len(image_paths)*100:.1f}%)')
print(f'Test:  {len(test_paths)}  ({len(test_paths)/len(image_paths)*100:.1f}%)')

## 3. Verificación de no-fuga entre splits

In [ ]:
train_set = set(train_paths)
val_set   = set(val_paths)
test_set  = set(test_paths)

tv_overlap  = train_set & val_set
ttest_overlap = train_set & test_set
vtest_overlap = val_set  & test_set

assert len(tv_overlap)    == 0, f'FUGA train/val: {len(tv_overlap)} imágenes'
assert len(ttest_overlap) == 0, f'FUGA train/test: {len(ttest_overlap)} imágenes'
assert len(vtest_overlap) == 0, f'FUGA val/test: {len(vtest_overlap)} imágenes'

print('Sin fuga de datos entre splits.')

## 4. Distribución por clase en cada split

In [ ]:
train_dist = Counter(train_labels)
val_dist   = Counter(val_labels)
test_dist  = Counter(test_labels)

clases = sorted(train_dist.keys())
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (split_name, dist) in zip(axes, [('Train', train_dist), ('Val', val_dist), ('Test', test_dist)]):
    valores = [dist.get(c, 0) for c in clases]
    bars = ax.bar(clases, valores, color='steelblue')
    ax.set_title(f'{split_name} ({sum(valores)} imágenes)', fontsize=12)
    ax.set_xlabel('Clase')
    ax.set_ylabel('Cantidad')
    ax.tick_params(axis='x', rotation=45)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                int(bar.get_height()), ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribución por clase en cada split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla numérica
df_dist = pd.DataFrame({'Train': train_dist, 'Val': val_dist, 'Test': test_dist}).fillna(0).astype(int)
df_dist.loc['TOTAL'] = df_dist.sum()
print(df_dist)

## 5. Guardado de CSVs

In [ ]:
out_dir = processed_data_dir()
out_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.DataFrame({'image_path': train_paths, 'label': train_labels})
val_df   = pd.DataFrame({'image_path': val_paths,   'label': val_labels})
test_df  = pd.DataFrame({'image_path': test_paths,  'label': test_labels})

train_csv = out_dir / 'train.csv'
val_csv   = out_dir / 'val.csv'
test_csv  = out_dir / 'test.csv'

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv,     index=False)
test_df.to_csv(test_csv,   index=False)

print(f'Guardados en {out_dir}:')
print(f'  train.csv: {len(train_df)} filas')
print(f'  val.csv:   {len(val_df)} filas')
print(f'  test.csv:  {len(test_df)} filas')

## 6. Verificación final

In [ ]:
print('=' * 55)
print('VERIFICACIÓN FINAL')
print('=' * 55)

for csv_path, expected_len in [(train_csv, len(train_df)), (val_csv, len(val_df)), (test_csv, len(test_df))]:
    assert csv_path.exists(), f'No existe: {csv_path}'
    df_check = pd.read_csv(csv_path)
    assert len(df_check) == expected_len, f'{csv_path.name}: {len(df_check)} filas, esperado {expected_len}'
    # Verificar que todos los archivos referenciados existen en disco
    missing = [p for p in df_check['image_path'] if not Path(p).exists()]
    if missing:
        print(f'  ADVERTENCIA {csv_path.name}: {len(missing)} rutas no existen en disco')
        print(f'  Ejemplo: {missing[0]}')
    else:
        print(f'  [OK] {csv_path.name}: {len(df_check)} filas, todas las rutas existen')

print()
print('Notebook 1.3 completado correctamente.')